[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-3_%EA%B3%A0%EC%9E%A5%EC%A7%84%EB%8B%A8_%EB%B6%84%EB%A5%98%EB%AA%A8%EB%8D%B8.ipynb)

# 2-3 · 고장진단 — 어떤 고장인지 분류하는 모델 만들기

**⏱ 110분** · 모듈 2: 센서 → 특징 → 이상탐지 → **② 고장진단**

## 🎯 이번 시간의 질문

> 경보가 울렸습니다. **어느 부품이, 얼마나** 나빠졌을까요?

2-2의 이상탐지는 "평소와 다르다"만 알려 주었고, 미세한 펌프 누설은 20%밖에 잡지 못했습니다.
하지만 과거 기록에 **"이 사이클은 누설 약함이었다"**는 정답(라벨)이 있다면, 정답을 보면서 **센서 특징 → 상태**의 관계를 학습할 수 있습니다.
이것이 **지도학습 분류(classification)**이고, CBM에서는 **고장진단**이라고 부릅니다.

| | 이상탐지 (2-2) | 고장진단 (2-3) |
|---|---|---|
| 학습 데이터 | 정상 데이터만 | 정상 + 여러 고장, **정답 포함** |
| 답하는 질문 | 평소와 다른가? | **어떤** 고장이고 **얼마나** 심한가? |
| 코드 | `model.fit(X)` | `model.fit(X, y)` |

**오늘의 목표:** 펌프 누설 상태(0 없음 · 1 약함 · 2 심함)를 센서 특징으로 진단하는 모델을 만들고, 다른 부품으로 확장합니다.

**이번 시간에 익힐 패턴:** P5 모델 세 줄(`fit` → `predict` → 평가) 반복 · 교차표(혼동행렬) · 특징 중요도 막대그래프

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')
train = data[data['split'] == 'train']
test = data[data['split'] == 'test']

# 입력으로 쓸 특징: 17개 센서 중 14개의 '평균' 특징
mean_cols = ['PS1_mean', 'PS2_mean', 'PS3_mean', 'PS4_mean', 'PS5_mean', 'PS6_mean', 'EPS1_mean',
             'FS1_mean', 'FS2_mean', 'TS1_mean', 'TS2_mean', 'TS3_mean', 'TS4_mean', 'VS1_mean']
print('학습:', len(train), '사이클 / 평가:', len(test), '사이클 / 입력 특징:', len(mean_cols), '개')

⚠️ **학습용과 평가용을 어떻게 나눴나** — 이 장치는 같은 상태 조합으로 여러 사이클을 연달아 돌렸기 때문에, 이웃한 사이클은 거의 쌍둥이입니다.
무작위로 나누면 쌍둥이가 학습용과 평가용에 나뉘어 들어가 **시험 문제를 미리 본 것**처럼 성능이 부풀려집니다.
그래서 **같은 상태 조합은 한쪽에만** 들어가도록 미리 나눠 두었습니다(`split` 열). 평가 사이클은 모델이 처음 보는 상태 조합입니다.

## 1. 입력(X)과 정답(y) 정하기

- **X** = 모델이 보는 입력 (센서 특징 14개)
- **y** = 맞혀야 할 정답 (펌프 누설 상태)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
X_train = train[mean_cols]
y_train = train['pump_leakage']
X_test = test[mean_cols]
y_test = test['pump_leakage']
print(X_train.shape, y_train.shape)
y_test.value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 학습 1,869개 × 특징 14개. 평가용은 누설 없음 90 · 약함 123 · 심함 123개입니다.

## 2. 결정나무 — 사람이 읽을 수 있는 규칙

결정나무(Decision Tree)는 "**유량이 6.5보다 작은가?**" 같은 **예/아니오 질문을 이어서** 답을 찾습니다. 질문과 기준값은 학습 데이터에서 자동으로 고릅니다.
`max_depth=2`는 질문을 최대 두 번만 하라는 뜻입니다.

**P5 · 모델 세 줄** — 만들기 → 학습(`fit`) → 예측(`predict`), 그리고 정확도로 평가합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

tree2 = DecisionTreeClassifier(max_depth=2, random_state=0)
tree2.fit(X_train, y_train)
pred2 = tree2.predict(X_test)
print('정확도:', round(accuracy_score(y_test, pred2) * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정확도 82.1%.

학습한 나무를 그림으로 봅니다. 각 칸의 첫 줄이 질문, `samples`는 그 칸에 도달한 학습 사이클 수, `value`는 상태별 개수, `class`는 그 칸의 판정입니다.

In [ ]:
# ▶ 그대로 실행하세요
plt.figure(figsize=(12, 5))
plot_tree(tree2, feature_names=mean_cols, class_names=['없음', '약함', '심함'],
          filled=True, impurity=False, fontsize=10)
plt.show()

💬 **결과 해설** — 나무가 스스로 고른 규칙은 이렇습니다.

```
유량 FS1_mean > 6.54            → 누설 없음
6.44 < 유량 FS1_mean ≤ 6.54    → 약한 누설
유량 FS1_mean ≤ 6.44            → 심한 누설
```

"누설이 심할수록 유량이 줄어든다" — 2-1 과제 1에서 눈으로 본 관계를 **숫자 기준까지 정해서** 찾아냈습니다. 이렇게 규칙을 사람이 읽고 검증할 수 있는 것이 결정나무의 큰 장점입니다.

### 무엇을 틀렸나 — 혼동행렬

정확도 하나로는 **어떤 실수**를 했는지 모릅니다. 실제 × 예측 교차표(혼동행렬)를 봅니다. 대각선이 맞힌 것, 나머지가 틀린 것입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
pd.crosstab(y_test, pred2, rownames=['실제'], colnames=['예측'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 가장 큰 실수는 **약한 누설(1) 123개 중 50개를 심한 누설(2)로** 본 것입니다. 기준값 하나(6.44)로는 두 상태를 깔끔히 가를 수 없습니다.
운영 관점에서는 실수의 **종류**가 중요합니다. "약함 → 심함" 오판은 정비를 조금 서두르게 할 뿐이지만, "심함 → 없음" 오판은 사고로 이어질 수 있습니다. 여기서는 심한 누설을 없음으로 본 경우는 0건입니다.

## 3. 더 깊은 나무, 그리고 숲(Random Forest)

질문을 네 번까지 허용하면 어떨까요? P5 세 줄을 **숫자 하나만 바꿔** 다시 씁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
tree4 = DecisionTreeClassifier(max_depth=4, random_state=0)
tree4.fit(X_train, y_train)
pred4 = tree4.predict(X_test)
print('정확도:', round(accuracy_score(y_test, pred4) * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 82.1% → **93.5%**. 질문이 늘면서 약함/심함을 더 잘 가릅니다. 대신 규칙이 복잡해져 사람이 읽기는 어려워집니다.

**랜덤 포레스트(Random Forest)** 는 결정나무 **100그루**를 만들어 **투표**로 결정합니다. 각 나무는 학습 데이터와 특징의 **일부만 무작위로** 보고 학습하기 때문에 서로 조금씩 다른 시각을 갖고, 한 나무의 실수를 다른 나무들이 보완합니다.
코드는 모델 이름만 다를 뿐 **똑같은 세 줄**입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(random_state=0)
forest.fit(X_train, y_train)
pred_rf = forest.predict(X_test)
print('정확도:', round(accuracy_score(y_test, pred_rf) * 100, 1), '%')
pd.crosstab(y_test, pred_rf, rownames=['실제'], colnames=['예측'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정확도 97.3%.

💬 **결과 해설** — 336개 중 9개만 틀렸습니다(심함 → 약함 8개, 없음 → 약함 1개). 2-2의 이상탐지(누설 탐지율 20%)와 비교하면 **정답 라벨로 학습한 효과**가 매우 큽니다.
세 모델을 정리하면:

| 모델 | 평가 정확도 | 장점 | 단점 |
|---|---|---|---|
| 결정나무 (깊이 2) | 82.1% | 규칙을 사람이 읽을 수 있음 | 복잡한 경계를 못 그림 |
| 결정나무 (깊이 4) | 93.5% | 여전히 그림으로 확인 가능 | 규칙이 길어짐 |
| 랜덤 포레스트 | 97.3% | 가장 정확, 안정적 | 100그루라 규칙을 직접 읽기 어려움 |

## 4. 모델은 어떤 센서를 중요하게 봤나 — 특징 중요도

랜덤 포레스트는 각 특징이 판정에 얼마나 기여했는지 `feature_importances_`로 알려 줍니다. 합계는 1입니다.

**P4 · 가로 막대그래프** — 중요도 순서로 정렬해 그립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
importance = pd.Series(forest.feature_importances_, index=mean_cols).sort_values()
plt.barh(importance.index, importance.values)
plt.xlabel('특징 중요도')
plt.title('펌프 누설 진단에 쓰인 특징')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 유량 `FS1_mean`(약 0.42)이 압도적으로 중요하고, 모터 전력 `EPS1_mean`(0.14), 압력 `PS3_mean`, `PS1_mean`이 뒤를 잇습니다.
펌프 안에서 기름이 새면 **내보내는 유량이 줄고**, 같은 압력을 유지하려고 **모터가 더 많은 전력**을 씁니다. 모델이 물리적으로 말이 되는 센서를 쓰고 있다는 뜻이므로 **모델을 믿을 근거**가 됩니다.
⚠️ 중요도는 "모델이 많이 참고한 정도"이지 원인·결과 관계를 증명하지는 않습니다.

---
## 5. 실습 과제 (45분)

모든 과제는 **P5 세 줄을 복사하지 말고 다시 입력**해 보세요. 바꿀 것은 정답 열(y)과 입력 특징(X)뿐입니다.

### 📝 과제 1 · 냉각기 진단
정답을 `cooler`로 바꿔 결정나무(깊이 2)와 랜덤 포레스트로 진단하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: y_train_c = train['cooler'], y_test_c = test['cooler'] 처럼 새 이름으로 정답을 만들고 1~3절의 코드를 다시 씁니다.
# (원래 y_train, y_test는 펌프 누설 정답이므로 덮어쓰지 않습니다. 과제 5에서 다시 씁니다.)


### 📝 과제 2 · 밸브 진단 — 이번에도 잘 될까?
정답을 `valve`(100 · 90 · 80 · 73)로 바꿔 결정나무(깊이 4)와 랜덤 포레스트의 정확도와 혼동행렬을 구하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 과제 1과 같은 방법. 정답만 y_train_v = train['valve'], y_test_v = test['valve'] 로.


✏️ **나의 답:** 정확도는? 네 가지 상태 중 아무거나 찍을 때(약 25%)와 비교하면? 왜 잘 안 될까요? (2-1 과제 2를 떠올려 보세요)

### 📝 과제 3 · 좋은 특징 하나 추가하기 (도전)
2-1 과제 3에서 만든 **"9~10초 압력 평균"** 특징을 추가하면 밸브 진단이 좋아질까요?

1. 원신호에서 특징 만들기 → `data['PS1_peak9'] = raw['PS1'][:, 900:1000].mean(axis=1)` (먼저 `raw = np.load('data/hydraulic_raw_4sensors.npz')`)
2. 새 열이 들어간 `data`에서 학습/평가를 다시 나누기 → `train2 = data[data['split'] == 'train']`, `test2 = ...`
3. 입력 특징 목록에 추가 → `cols_plus = mean_cols + ['PS1_peak9']` 하고 과제 2를 반복

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 예전 train, test에는 새 열이 없으므로 반드시 train2, test2를 새로 만들어 씁니다.


✏️ **나의 답:**

### 📝 과제 4 · 축압기 진단
정답을 `accumulator`(130 · 115 · 100 · 90)로 바꿔 결정나무(깊이 2)와 랜덤 포레스트를 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 과제 1과 같은 방법. 정답 열만 'accumulator'로, 입력은 X_train, X_test.


✏️ **나의 답:** 두 모델의 차이가 큰 이유는 무엇일까요? 혼동행렬에서 틀린 것은 주로 어떤 상태끼리인가요?

### 📝 과제 5 · 나무를 깊게 할수록 좋아질까? (도전)
펌프 누설 진단에서 `max_depth`를 1부터 10까지 바꾸며 **학습 정확도와 평가 정확도**를 같은 그래프에 그리세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 빈 리스트 두 개를 만들고 for depth in range(1, 11): 안에서 학습 → 두 정확도를 append → plt.plot 두 번


---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 지도학습 분류 | 정답(y)이 있는 과거 데이터로 `fit(X, y)` → 새 사이클의 상태를 `predict(X)` |
| 결정나무 | 예/아니오 질문의 연속. 규칙을 **읽을 수 있다** (유량 ≤ 6.44 → 심한 누설) |
| 랜덤 포레스트 | 나무 여러 그루의 투표. 대개 더 정확하지만 규칙을 직접 읽기 어렵다 |
| 평가 | 정확도 + **혼동행렬**로 어떤 실수를 하는지 본다. 학습용이 아닌 **처음 보는 데이터**로 |
| 특징 중요도 | 모델이 물리적으로 말이 되는 센서를 쓰는지 확인 |
| 가장 큰 교훈 | 입력에 정보가 없으면 어떤 모델도 못 맞힌다. **좋은 특징 > 복잡한 모델** (밸브 43% → 98%) |

다음: **2-4 · 종합 실습 — 진단 결과로 경보와 정비 판단하기**